In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../../data/raw/anime.csv")

In [3]:
df.columns

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='object')

In [ ]:
feature_list = ["anime_id","Name","English name","Score","Genres","Synopsis","Type","Studios","Source","Popularity","Scored By","Image URL","Aired","Premiered","Status","Rank"]

In [5]:
df = df[feature_list]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   anime_id      24905 non-null  int64 
 1   Name          24905 non-null  object
 2   English name  24905 non-null  object
 3   Score         24905 non-null  object
 4   Genres        24905 non-null  object
 5   Synopsis      24905 non-null  object
 6   Type          24905 non-null  object
 7   Studios       24905 non-null  object
 8   Source        24905 non-null  object
 9   Popularity    24905 non-null  int64 
 10  Scored By     24905 non-null  object
 11  Image URL     24905 non-null  object
dtypes: int64(2), object(10)
memory usage: 2.3+ MB


In [7]:
df

,anime_id,Name,English name,Score,Genres,Synopsis,Type,Studios,Source,Popularity,Scored By,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,8.75,"Action, Award Winning, Sci-Fi","Crime is timeless. By the year 2071, humanity ...",TV,Sunrise,Original,43,914193.0,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,8.38,"Action, Sci-Fi","Another day, another bounty—such is the life o...",Movie,Bones,Original,602,206248.0,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,8.22,"Action, Adventure, Sci-Fi","Vash the Stampede is the man with a $$60,000,0...",TV,Madhouse,Manga,246,356739.0,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,7.25,"Action, Drama, Mystery, Supernatural",Robin Sena is a powerful craft user drafted in...,TV,Sunrise,Original,1795,42829.0,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,6.94,"Adventure, Fantasy, Supernatural",It is the dark century and the people are suff...,TV,Toei Animation,Manga,5126,6413.0,https://cdn.myanimelist.net/images/anime/7/215...
...,...,...,...,...,...,...,...,...,...,...,...,...
24900,55731,Wu Nao Monu,UNKNOWN,UNKNOWN,"Comedy, Fantasy, Slice of Life",No description available for this anime.,ONA,UNKNOWN,Web manga,24723,UNKNOWN,https://cdn.myanimelist.net/images/anime/1386/...
24901,55732,Bu Xing Si: Yuan Qi,Blader Soul,UNKNOWN,"Action, Adventure, Fantasy",No description available for this anime.,ONA,UNKNOWN,Web novel,0,UNKNOWN,https://cdn.myanimelist.net/images/anime/1383/...
24902,55733,Di Yi Xulie,The First Order,UNKNOWN,"Action, Adventure, Fantasy, Sci-Fi",No description available for this anime.,ONA,UNKNOWN,Web novel,0,UNKNOWN,https://cdn.myanimelist.net/images/anime/1130/...
24903,55734,Bokura no Saishuu Sensou,UNKNOWN,UNKNOWN,UNKNOWN,A music video for the song Bokura no Saishuu S...,Music,UNKNOWN,Original,0,UNKNOWN,https://cdn.myanimelist.net/images/anime/1931/...


In [8]:
no_description = df.query("Synopsis == 'No description available for this anime.'")
df = df.drop(no_description.index)

In [9]:
#show rows with UNKNOWN values in any of the columns expect English name
drop_unknowns = df.drop(columns={"English name"})
drop_unknowns = drop_unknowns[drop_unknowns == "UNKNOWN"].dropna(how="all").index
clean_df = df.drop(drop_unknowns)

In [13]:
clean_df[clean_df["English name"] == "Death Note"]

,anime_id,Name,English name,Score,Genres,Synopsis,Type,Studios,Source,Popularity,Scored By,Image URL
1393,1535,Death Note,Death Note,8.62,"Supernatural, Suspense","Brutal murders, petty thefts, and senseless vi...",TV,Madhouse,Manga,2,2619479.0,https://cdn.myanimelist.net/images/anime/9/945...


In [11]:
# create a function that takes a name and looks for the Name and English name columns and return the row if it finds a match in either of the columns. the name can have a partial match. Yeah
def find_anime(name):
    name = name.lower()
    for index, row in clean_df.iterrows():
        if name in row["Name"].lower() or name in row["English name"].lower():
            return row
    return None
find_anime("alchemist")

anime_id                                                      121
Name                                          Fullmetal Alchemist
English name                                  Fullmetal Alchemist
Score                                                        8.11
Genres           Action, Adventure, Award Winning, Drama, Fantasy
Synopsis        Edward Elric, a young, brilliant alchemist, ha...
Type                                                           TV
Studios                                                     Bones
Source                                                      Manga
Popularity                                                     76
Scored By                                                868434.0
Image URL       https://cdn.myanimelist.net/images/anime/10/75...
Name: 100, dtype: object

In [12]:
#turn the clean_df into parquet
clean_df.to_parquet("../../data/processed/anime.parquet", index=False)

In [15]:
#show the unique genre names in the clean_df. currently its a string of genres separated by commas. we need to split them and get the unique values
def parse_genre_set(genre_str):
    if pd.isna(genre_str) or genre_str.strip() == "":
        return set()
    return set(g.strip().lower() for g in genre_str.split(",") if g.strip() != "")

clean_df["Genres"].apply(parse_genre_set).explode().unique()

array(['sci-fi', 'action', 'award winning', 'adventure', 'mystery',
       'drama', 'supernatural', 'fantasy', 'sports', 'romance', 'comedy',
       'slice of life', 'suspense', 'ecchi', 'gourmet', 'avant garde',
       'horror', 'girls love', 'boys love', 'hentai', 'erotica'],
      dtype=object)